# 了解接口类

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install gradio

In [ ]:
import numpy as np
import gradio as gr


def reverse_audio(audio):
    if audio is None:
        return None
    sr, data = audio
    reversed_audio = (sr, np.flipud(data))
    return reversed_audio


mic = gr.Audio(sources="microphone", type="numpy", label="Speak here...")
gr.Interface(reverse_audio, mic, "audio").launch()

In [ ]:
import numpy as np
import gradio as gr

notes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def generate_tone(note, octave, duration):
    sr = 48000  # 采样率：每秒 48000 个采样点（音频标准之一）

    # 计算目标音符与 A4（440Hz）的半音距离
    # octave - 4：当前八度与第4八度的差，每差1个八度 = 12个半音
    # note - 9：当前音符与 A（索引9）的半音差
    tones_from_a4 = 12 * (octave - 4) + (note - 9)
    a4_freq = 440

    # 十二平均律公式：f = 440 * 2^(n/12)
    # n 为与 A4 的半音数，每个半音频率比为 2^(1/12) ≈ 1.0595
    frequency = a4_freq * 2 ** (tones_from_a4 / 12)

    duration = int(duration)  # 确保时长为整数秒

    # 生成时间轴：从 0 到 duration，共 duration * sr 个点
    # 每个点代表一个采样时刻（单位：秒）
    audio = np.linspace(0, duration, duration * sr)

    # 生成正弦波：sin(2π * f * t)
    # 振幅 20000 控制音量（int16 范围 -32768~32767，20000 约为 60% 音量）
    # 转为 int16 是 PCM 音频的标准格式
    audio = (20000 * np.sin(audio * (2 * np.pi * frequency))).astype(np.int16)

    # 返回 (采样率, 音频数据) 元组，Gradio Audio 组件的标准格式
    return (sr, audio)


gr.Interface(
    generate_tone,
    [
        gr.Dropdown(notes, type="index"),
        gr.Slider(minimum=4, maximum=6, step=1),
        gr.Number(value=1, label="Duration in seconds"),
    ],
    "audio",
).launch()

In [ ]:
from transformers import pipeline
import gradio as gr

model = pipeline("automatic-speech-recognition")


def transcribe_audio(mic=None, file=None):
    if mic is not None:
        audio = mic
    elif file is not None:
        audio = file
    else:
        return "You must either provide a mic recording or a file"
    transcription = model(audio)["text"]
    return transcription


gr.Interface(
    fn=transcribe_audio,
    inputs=[
        gr.Audio(sources="microphone", type="filepath"),
        gr.Audio(sources="upload", type="filepath"),
    ],
    outputs="text",
).launch()